# Kinetiq Studio — Google Colab GPU Worker (L4 / A100)
Pull-based ComfyUI worker using Colab Secrets (`API_BASE_URL`, `WORKER_TOKEN`, optional `HF_TOKEN`).
Uploads finished 1080p/4K masters and 540p proxies directly to your Google Drive via 8 MiB resumable chunks.

In [ ]:
# Cell 1: Load Secrets & Auto-Tune VRAM Flags
import os

import torch
from google.colab import userdata

os.environ['API_BASE_URL'] = userdata.get('API_BASE_URL')
os.environ['WORKER_TOKEN'] = userdata.get('WORKER_TOKEN')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print('Using public models without an HF token.')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

PROFILE = 'ltx25_pro'  # options: 'ltx25_pro' | 'wan22_fast' | 'upscale_4k'

# Use the assigned accelerator and retain PROFILE so queued jobs remain eligible.
if not torch.cuda.is_available():
    raise RuntimeError('Attach an L4 or A100 GPU runtime before starting the worker.')
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 2**30)
low_vram = 'L4' in gpu_name.upper().split() or vram_gb < 38
if low_vram:
    comfy_flags = ['--lowvram', '--fp8_e4m3fn-unet']
else:
    comfy_flags = ['--highvram' if vram_gb >= 70 else '--normalvram']
print(f'GPU: {gpu_name} ({vram_gb} GiB) | Flags: {comfy_flags} | Job profile: {PROFILE}')

In [ ]:
%%bash
# Cell 2: Install Headless ComfyUI, Custom Nodes & kinetiq-agent-core
set -e
pip install -q hf_transfer huggingface_hub httpx websocket-client psutil

if [ ! -d '/content/ComfyUI' ]; then
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
  pip install -q -r /content/ComfyUI/requirements.txt
fi

if [ ! -d /content/kinetiq-studio ]; then
  git clone --depth 1 https://github.com/TeddyQyuon/kinetiq-studio.git /content/kinetiq-studio
fi

cd /content/ComfyUI/custom_nodes
for repo in \
  'https://github.com/Lightricks/ComfyUI-LTXVideo.git' \
  'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git' \
  'https://github.com/1038lab/ComfyUI-FlashVSR.git' \
  'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git' \
  'https://github.com/city96/ComfyUI-GGUF.git'; do
  name=$(basename "$repo" .git)
  if [ ! -d "$name" ]; then
    git clone --depth 1 "$repo"
  fi
  if [ -f "$name/requirements.txt" ]; then pip install -q -r "$name/requirements.txt"; fi
done

In [ ]:
# Cell 3: Fast Profile-Aware Model Bootstrap via hf_transfer
from pathlib import Path

from huggingface_hub import hf_hub_download

MODELS = Path('/content/ComfyUI/models')

def fetch(repo_id: str, filename: str, subdir: str):
    dest = MODELS / subdir
    dest.mkdir(parents=True, exist_ok=True)
    path = Path(hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(dest)))
    name = dest / path.name
    if path != name and not name.exists():
        name.symlink_to(path)

if PROFILE in {"ltx25_pro", "wan22_fast"}:
    fetch('Comfy-Org/Qwen-Image', 'split_files/diffusion_models/qwen_image_fp8_e4m3fn.safetensors', 'diffusion_models')
    fetch('Comfy-Org/Qwen-Image', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', 'text_encoders')
    fetch('Comfy-Org/Qwen-Image', 'split_files/vae/qwen_image_vae.safetensors', 'vae')

if PROFILE == 'wan22_fast':
    base = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
    fetch(base, 'split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors', 'diffusion_models')
    fetch(base, 'split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors', 'diffusion_models')
    fetch(base, 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors', 'loras')
    fetch(base, 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors', 'loras')
    fetch(base, 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders')
    fetch(base, 'split_files/vae/wan_2.1_vae.safetensors', 'vae')
elif PROFILE == 'ltx25_pro':
    fetch('unsloth/LTX-2.5-FP8', 'ltx-2.5-22b-distilled-transformer-fp8.safetensors', 'diffusion_models')
    fetch('Lightricks/LTX-2.5', 'gemma4-12b-with-proj-ltx-2.5-bf16.safetensors', 'text_encoders')
    fetch('Lightricks/LTX-2.5', 'ltx-2.5-video-vae-bf16.safetensors', 'vae')
    fetch('Lightricks/LTX-2.5', 'ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors', 'latent_upscale_models')
elif PROFILE == 'upscale_4k':
    fetch('numz/SeedVR2_comfyUI', 'seedvr2_ema_3b_fp8_e4m3fn.safetensors', 'SEEDVR2')
    fetch('numz/SeedVR2_comfyUI', 'ema_vae_fp16.safetensors', 'SEEDVR2')
print('Model bootstrap complete for profile:', PROFILE)

In [ ]:
# Cell 4: Start ComfyUI before registering or leasing work
import subprocess
import time
from contextlib import suppress

import httpx

with open('/content/comfy.log', 'w') as log:
    comfy_proc = subprocess.Popen(
        ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188',
         *comfy_flags, '--disable-auto-launch', '--preview-method', 'none'],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

for _ in range(90):
    if comfy_proc.poll() is not None:
        raise RuntimeError('ComfyUI exited during startup. See /content/comfy.log.')
    with suppress(httpx.RequestError):
        response = httpx.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if response.is_success:
            break
    time.sleep(2)
else:
    raise TimeoutError('ComfyUI startup timed out. See /content/comfy.log.')

if PROFILE == 'upscale_4k':
    nodes = httpx.get('http://127.0.0.1:8188/object_info', timeout=30)
    nodes.raise_for_status()
    required = {'AILab_FlashVSR', 'SeedVR2LoadDiTModel', 'SeedVR2LoadVAEModel', 'SeedVR2VideoUpscaler',
                'VHS_LoadVideo', 'VHS_VideoInfoLoaded', 'VHS_VideoCombine'}
    if missing := required.difference(nodes.json()):
        raise RuntimeError(f'Missing upscale nodes: {sorted(missing)}. See /content/comfy.log.')
print('ComfyUI ready on 127.0.0.1:8188')

In [ ]:
# Cell 5: Launch Pull-Based GPU Worker (install kinetiq-agent-core & runner.py from your repo)
!pip install -q -e /content/kinetiq-studio/workers/agent-core
!python /content/kinetiq-studio/workers/gpu/runner.py --profile $PROFILE --provider google_colab --idle-minutes 15